# RSM456 · Exercise 2 — Who are our customers? Segmentation with k-means

**Session 2 · Wednesday, January 20, 2027 · ~45 minutes in class**

You are an analyst at a large online travel platform. Your manager hands you a sample of 1,000 users and asks:
*"Who are our customers? Can we group them into a few segments that would each deserve a different marketing approach?"*

By the end of this exercise you will have:
1. explored the behavioural data and spotted the obvious patterns,
2. run k-means clustering, chosen a number of clusters, and profiled each segment,
3. named the segments and proposed one marketing action for each.

---
### How to work in this notebook (read once)

- **First: File → Save a copy in Drive.** You are looking at the course's read-only copy. Work in your own copy.
- Run cells from top to bottom with **Shift+Enter** (or ▶ on the left of a cell). A cell that hasn't been run shows `[ ]`; after running it shows a number.
- Cells marked **✍️ Your turn** are where you write — either code or a few sentences in a text cell. The rest is provided.
- **AI is allowed and encouraged.** Click the ✨ *Gemini* button (top right) to ask about any cell, or paste a cell into ChatGPT/Claude. Good questions: *"Explain what this cell does"*, *"Why standardize before k-means?"*, *"I got this error, fix it"*. You must be able to explain everything you submit — expect to be asked in class.
- **Submit before the end of class:** `Runtime → Run all`, then `File → Download → Download .ipynb`, then upload the file to the **Exercise 2** assignment on Quercus. Completion counts; perfection doesn't.


## 0 · Setup (run this cell first)

Everything needed is already installed in Colab. This cell loads the libraries, fixes the random seed so your results match your neighbour's, and reads the data straight from the course data site — nothing to upload.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

SEED = 42
np.random.seed(SEED)
pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid")

# Course data is hosted on a public site. The address below points at the RSM456 course data site.
DATA_URL = "https://raw.githubusercontent.com/zacharyzzhong/rsm456-2027/main/Exercises/data/clustering_exercise_1000users.csv"

try:
    users = pd.read_csv(DATA_URL)
except Exception as e:                      # fallback: upload the file by hand
    print("Could not read from the course site; please upload clustering_exercise_1000users.csv")
    from google.colab import files           # only exists inside Colab
    uploaded = files.upload()
    users = pd.read_csv(next(iter(uploaded)))

print(f"Loaded {users.shape[0]:,} users and {users.shape[1]} variables.")
users.head()

**The variables**

| Variable | Meaning |
|---|---|
| `female` | 1 = female, 0 = male |
| `agegroup_level` | age bracket (1 = youngest … 5 = oldest) |
| `localcity_level_num` | city tier of the user's home city (lower = bigger city) |
| `device_ios` | 1 = iPhone, 0 = Android |
| `vip_level` | loyalty tier (higher = more premium) |
| `days_reg_july10` | days since the account was created (tenure) |
| `login_minutes_total` | total minutes spent in the app |
| `N_query_total` | number of searches |
| `N_clicks_keyword_total` | number of clicks on sponsored keyword results |
| `N_order_flight_total` | flight bookings |
| `N_order_hotel_total` | hotel bookings |


## Part A · Look before you cluster (≈10 min)

Clustering finds groups in whatever you feed it. If you don't know what the data look like, you won't know whether the groups are real. Three quick checks: how skewed is engagement, how many users ever buy, and does time in the app translate into bookings?

In [ ]:
users.describe().round(2).T

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for ax, col in zip(axes, ["login_minutes_total", "N_order_flight_total", "N_order_hotel_total"]):
    sns.histplot(users[col], bins=30, ax=ax)
    ax.set_title(col)
plt.tight_layout()

In [ ]:
share = {
    "at least one flight order": (users["N_order_flight_total"] > 0).mean(),
    "at least one hotel order":  (users["N_order_hotel_total"]  > 0).mean(),
    "both":                      ((users["N_order_flight_total"] > 0) & (users["N_order_hotel_total"] > 0)).mean(),
    "neither":                   ((users["N_order_flight_total"] == 0) & (users["N_order_hotel_total"] == 0)).mean(),
}
pd.Series(share, name="share of users").map("{:.1%}".format)

In [ ]:
users["orders_total"] = users["N_order_flight_total"] + users["N_order_hotel_total"]

fig, ax = plt.subplots(figsize=(6, 4))
sns.scatterplot(data=users, x="login_minutes_total", y="orders_total", alpha=0.4, ax=ax)
ax.set(xlabel="Total minutes in app", ylabel="Total orders (flight + hotel)",
       title=f"Correlation = {users['login_minutes_total'].corr(users['orders_total']):.2f}")
plt.show()

### ✍️ Your turn (A)
Double-click this cell and write 2–3 sentences: What stands out? Is engagement evenly spread or concentrated in a few users? Can you already *see* a group or two in the plots?

> *Your answer here.*


## Part B · k-means clustering (≈20 min)

Two decisions before pressing the button:

1. **Which variables define a segment?** We cluster on *behaviour and relationship* (engagement, purchases, tenure, loyalty tier) and keep *demographics* aside to profile the segments afterwards. Segments defined by what people do are the ones you can act on; demographics tell you who is in them.
2. **Standardize.** `login_minutes_total` runs into the thousands, `N_order_flight_total` tops out at 4. Without putting every variable on the same scale, k-means would simply sort users by minutes.

In [ ]:
cluster_vars = ["login_minutes_total", "N_query_total", "N_clicks_keyword_total",
                "N_order_flight_total", "N_order_hotel_total", "days_reg_july10", "vip_level"]
profile_vars = ["female", "agegroup_level", "localcity_level_num", "device_ios"]

scaler = StandardScaler()
X = scaler.fit_transform(users[cluster_vars])
print("Standardized matrix:", X.shape, "| column means ≈", X.mean(axis=0).round(2), "| sds ≈", X.std(axis=0).round(2))

### B1 · Start with k = 4

In [ ]:
K = 4
km = KMeans(n_clusters=K, random_state=SEED, n_init=10)
users["cluster"] = km.fit_predict(X)

print("Cluster sizes:")
print(users["cluster"].value_counts().sort_index().to_string())

cluster_means = users.groupby("cluster")[cluster_vars].mean().round(1)
cluster_means

A table of means is hard to read across variables with different units. The **profile chart** below shows each cluster's standardized mean: bars above zero mean "above the average user", below zero "below average".

In [ ]:
profile = pd.DataFrame(X, columns=cluster_vars).assign(cluster=users["cluster"]).groupby("cluster").mean()

fig, ax = plt.subplots(figsize=(10, 4.5))
profile.T.plot(kind="bar", ax=ax, width=0.8)
ax.axhline(0, color="black", lw=0.8)
ax.set(ylabel="standardized mean (0 = average user)", title=f"Segment profiles, k = {K}")
ax.legend(title="cluster", ncol=K)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()

### B2 · How many clusters? The elbow plot

k-means will happily give you any number of clusters you ask for. The elbow plot shows how much *within-cluster* variation is left for each k. Past the "elbow", extra clusters buy little — but interpretability matters as much as the plot.

In [ ]:
inertia = {}
for k in range(2, 9):
    inertia[k] = KMeans(n_clusters=k, random_state=SEED, n_init=10).fit(X).inertia_

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(list(inertia), list(inertia.values()), marker="o")
ax.set(xlabel="number of clusters k", ylabel="within-cluster sum of squares", title="Elbow plot")
plt.show()

### ✍️ Your turn (B)
1. In the code cell below, change `K` to **3** and then **5**, re-run B1 and the profile chart, and compare. Which k gives segments you could describe in one sentence each?
2. Pick your k, set it below, and re-run B1 so the rest of the notebook uses your choice.

*Tip: ask Gemini "Why do the cluster labels change order when I change k?" — the answer matters for how you talk about segments.*

In [ ]:
# ✍️ Set your chosen number of clusters here, then re-run the B1 cell and the profile chart above.
K = 4   # <- change me, try 3 and 5 first

### ✍️ Your turn (B, continued) — name the segments
Fill in the table (double-click to edit). A good name says what the segment *does*, not just how big it is.

| Cluster | Size | What they do (1 line) | Segment name |
|---|---|---|---|
| 0 | | | |
| 1 | | | |
| 2 | | | |
| 3 | | | |


## Part C · So what? (≈15 min)

A segmentation is only useful if you would *do something different* for each segment. First, who is in each cluster (demographics we held back), then the decision.

In [ ]:
demo = users.groupby("cluster")[profile_vars].mean()
demo["n_users"] = users["cluster"].value_counts().sort_index()
demo["share_of_orders"] = users.groupby("cluster")["orders_total"].sum() / users["orders_total"].sum()
demo.round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
demo["n_users"].plot(kind="bar", ax=axes[0], title="Users per cluster")
demo["share_of_orders"].plot(kind="bar", ax=axes[1], title="Share of all orders", color="darkorange")
for ax in axes: ax.set_xlabel("cluster")
plt.tight_layout()

### ✍️ Your turn (C)
Answer in this cell (3–6 sentences total):

1. **One marketing action per segment** — e.g. re-engagement campaign, loyalty upgrade offer, cross-sell hotels to flight-only bookers, leave alone. Say why.
2. **Which segment has the highest value today, and which is the biggest growth opportunity?** Those are usually different segments.
3. **One limitation** of this segmentation you would flag to your manager.

> *Your answer here.*


## Stretch (optional, after class)

- Add the four demographic variables to `cluster_vars` and re-run. Do the segments change? Which basis would you defend in the session-2 terms (actionable, accessible, substantial)?
- Compute the **silhouette score** for k = 2…8 (`from sklearn.metrics import silhouette_score`) and compare with the elbow plot.
- k-means assumes round, similar-sized clusters. Try `AgglomerativeClustering` (hierarchical) on the same `X` and see whether the picture holds.


## Submit

1. `Runtime → Run all` and check that every cell runs without error.
2. `File → Download → Download .ipynb`.
3. Upload the file to the **Exercise 2** assignment on Quercus **before the end of this session**. (The assignment closes when class ends.)
4. Optional: `Share → Anyone with the link → Viewer`, and paste the link in the Exercise 2 discussion thread if you'd like your work shown in class.

*AI-use note: this exercise allows any AI tool. There is nothing to declare here — but in the team project you will add a short AI-use statement at the top of your notebook.*
